# Motif v2 — resumable free-Colab training

This notebook trains the compact category-aware motif encoder/decoder. Each run performs at most 5,000 optimizer steps and saves an atomic checkpoint to Google Drive. Rerun the training cell after a disconnect to continue. The final cells evaluate the complete official validation/test splits, export fixed held-out examples, and build the GitHub Release assets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/Tanmay-22/motif-piano-generator.git'
PROJECT = Path('/content/motif-piano-generator')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('The dependency install must not replace Colab CUDA torch.')
print('Project ready at', PROJECT)
print('Using torch', torch.__version__, 'with', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/motif-piano-v2')
OUTPUT_DIR = DRIVE_ROOT / 'checkpoints'
CACHE_DIR = DRIVE_ROOT / 'note-cache'
EXAMPLES_DIR = DRIVE_ROOT / 'examples'
RELEASE_DIR = DRIVE_ROOT / 'model-v2.0.0-release'
DATA_DIR = Path('/content/maestro-data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT_DIR)
print('Existing latest checkpoint:', (OUTPUT_DIR / 'latest.pt').exists())


## Train or resume

The first run downloads MAESTRO and creates one compressed note cache per official split. Later runs automatically load `latest.pt`. The three source periods are sampled with equal probability despite their different dataset sizes.


In [ ]:
command = [
    sys.executable, '-m', 'training.train_v2',
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--resume', 'auto',
    '--max-steps', '30000',
    '--session-steps', '5000',
    '--batch-size', '16',
    '--gradient-accumulation', '4',
    '--evaluate-every', '1000',
    '--save-every', '500',
    '--amp',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json

metrics_path = OUTPUT_DIR / 'metrics-v2.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    progress = metrics['progress']
    print('Global step:', progress['global_step'])
    print('Best validation loss:', progress['best_validation_loss'])
    if progress['history']:
        print('Latest validation:', json.dumps(progress['history'][-1], indent=2))
else:
    print('No validation report yet. Training must reach the first 1,000 steps.')


## Final held-out evaluation

Run this only after training finishes or early stopping selects the best checkpoint. This intentionally evaluates every batch in the official validation and test splits, so it takes longer than the periodic 50-batch checks. It also compares validation loss with a freshly initialized model.


In [ ]:
best_checkpoint = OUTPUT_DIR / 'conditioned-v2-best.pt'
if not best_checkpoint.exists():
    raise FileNotFoundError('No best checkpoint exists yet. Continue training first.')
evaluation_command = [
    sys.executable, '-m', 'training.train_v2',
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--evaluate-only', str(best_checkpoint),
    '--evaluation-max-batches', '0',
    '--batch-size', '16',
    '--amp',
]
subprocess.run(evaluation_command, check=True)
evaluation_path = OUTPUT_DIR / 'evaluation-v2.json'
evaluation = json.loads(evaluation_path.read_text())
print(json.dumps(evaluation, indent=2))
failed_gates = [name for name, passed in evaluation['quality_gates'].items() if not passed]
if failed_gates:
    raise RuntimeError('Do not release yet. Continue training; failed gates: ' + ', '.join(failed_gates))
print('All release quality gates passed.')


## Fixed held-out listening examples

Export one deterministic test motif, its real MAESTRO continuation, and the model continuation for each of the three period categories. These files are MAESTRO-derived and belong in the non-commercial model release, not Git history.


In [ ]:
examples_command = [
    sys.executable, '-m', 'training.export_v2_examples',
    '--checkpoint', str(best_checkpoint),
    '--data-dir', str(DATA_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(EXAMPLES_DIR),
    '--count', '3',
    '--duration-seconds', '10',
    '--temperature', '0.9',
    '--seed', '42',
    '--device', 'cuda',
]
subprocess.run(examples_command, check=True)
print((EXAMPLES_DIR / 'examples-v2.json').read_text())


## Build the public release bundle

The packager rechecks the full-split quality gates, validates that the checkpoint loads, computes its SHA-256 checksum, includes the license/evaluation/examples, and creates a ZIP in Google Drive.


In [ ]:
release_command = [
    sys.executable, '-m', 'scripts.prepare_v2_release',
    '--checkpoint', str(best_checkpoint),
    '--evaluation', str(evaluation_path),
    '--examples-dir', str(EXAMPLES_DIR),
    '--output-dir', str(RELEASE_DIR),
    '--release-tag', 'model-v2.0.0',
]
subprocess.run(release_command, check=True)
release_manifest = json.loads((RELEASE_DIR / 'release-manifest-v2.json').read_text())
print('Checkpoint SHA-256:', release_manifest['checkpoint']['sha256'])
print('\nCopy these values into Render:')
print((RELEASE_DIR / 'render-env-v2.txt').read_text())
print('Release files:', [str(path.relative_to(RELEASE_DIR)) for path in RELEASE_DIR.rglob('*') if path.is_file()])


## Download the release assets

Download all three files. Upload `conditioned-v2-best.pt` as its own GitHub Release asset because Render uses its direct URL. The ZIP is the complete reproducibility bundle, and `render-env-v2.txt` contains the exact deployment values. Never put the checkpoint or MAESTRO MIDI files in a normal Git commit.


In [ ]:
from google.colab import files
files.download(str(RELEASE_DIR / 'conditioned-v2-best.pt'))
files.download(str(RELEASE_DIR.with_suffix('.zip')))
files.download(str(RELEASE_DIR / 'render-env-v2.txt'))
